In [1]:
%cd ../
%ls

/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/Users/delapazm/Desktop/wellcome_academic_graph_toolkit
ID_All_Countries/   README.md           environment.yml     pyproject.toml
ID_LMIC_Countries/  careers/            geographies/        venv/
ID_LMIC_Countries2/ dist/               idr/                wag_toolkit/
Makefile            edges_all.json      nodes_all.json


In [2]:
from wag_toolkit.locations import Locations
import pandas as pd
import json
from collections import Counter
from dotenv import load_dotenv
import awswrangler as wr

# Load environment variables from .env file
load_dotenv()

True

In [ ]:
# grants_ref =  pd.read_csv('geographies/CH_files/Phase 2 Master Tagging Spreadsheet(GRANTS - Tagging).csv',
#                          encoding='latin1')
grants_ref =  wr.s3.read_csv('s3://datalabs-data/funding_impact_measures/cnfectious_disease/Phase_2_Master_Tagging_Spreadsheet(GRANTS-Tagging).csv', encoding='latin1')

grants_ids = list(set(grants_ref['Grant Reference'].to_list()))

In [4]:
from dotenv import load_dotenv
import neo4j
import os

# Load environment variables from .env file
load_dotenv()

driver = neo4j.GraphDatabase.driver(
    os.environ["NEO4J_BOLT_URL"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
)

query = """MATCH (g:Grant)-[LINKED_TO]->(p:Publication) WHERE g.original_source_id IN $grant_ids 
           RETURN g.original_source_id as grant_reference, p.dimensions_publication_id as publication_id"""

with driver.session() as session:
    result = session.run(query, grant_ids=grants_ids)  # Correctly pass the list
    id_grants_df = pd.DataFrame([dict(record) for record in result])

id_grants_df.head()

,grant_reference,publication_id
0,219622/Z/19/Z,pub.1149152302
1,219622/Z/19/Z,pub.1152015571
2,219622/Z/19/Z,pub.1150059899
3,219622/Z/19/Z,pub.1148976221
4,219622/Z/19/Z,pub.1155893525


In [5]:
dummy_query = """MATCH (r:Researcher)-[a:AUTHORED]->(p:Publication) RETURN * LIMIT 1"""
loc = Locations(dummy_query)

pub_ids = list(set(id_grants_df['publication_id'].to_list()))

query = """MATCH (r:Researcher)-[a:AUTHORED]->(p:Publication)
            WHERE p.dimensions_publication_id IN {}
            RETURN p.dimensions_publication_id AS dimensions_publication_id, p.year AS year,
                a.institutions AS grid_id"""
loc.lookup_query(query=query, lookup=pub_ids)

100%|██████████| 1/1 [00:01<00:00,  1.53s/it]


In [6]:
data =pd.DataFrame(loc.data)
data = data.explode("grid_id").dropna()
data.head()

,dimensions_publication_id,year,grid_id
0,pub.1143316090,2021.0,grid.264727.2
1,pub.1143316090,2021.0,['grid.7836.a']
2,pub.1143316090,2021.0,grid.301713.7
3,pub.1143316090,2021.0,grid.7836.a
4,pub.1143316090,2021.0,grid.4714.6


In [7]:
loc._clean_grid_ids()
loc.extract_edges()
loc.extract_locations("country")

loc.convert_edges()
loc.calculate_adjacency_matrices()

100%|██████████| 2/2 [00:00<00:00,  7.50it/s]
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:186: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:186: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:186: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep cu

2021.0
2018.0
2020.0
2022.0
2023.0
2019.0
2017.0
2015.0
2016.0
2014.0
2013.0


/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:186: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:186: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:186: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.


In [8]:
import numpy as np
loc.adjacency_matrices = {np.int64(key): value for key, value in loc.adjacency_matrices.items()}

In [10]:
countries_class = pd.read_excel('geographies/CH_files/CLASS.xlsx', sheet_name='List of economies')
lmic_list = countries_class[countries_class['Income group'].isin(['Low income', 'Lower middle income', 'Upper middle income'])]
lmic_list.head()

,Economy,Code,Region,Income group,Lending category
0,Afghanistan,AFG,South Asia,Low income,IDA
1,Albania,ALB,Europe & Central Asia,Upper middle income,IBRD
2,Algeria,DZA,Middle East & North Africa,Upper middle income,IBRD
5,Angola,AGO,Sub-Saharan Africa,Lower middle income,IBRD
7,Argentina,ARG,Latin America & Caribbean,Upper middle income,IBRD


In [12]:
lmic_list['Economy'] = lmic_list['Economy'].replace("Côte d’Ivoire", "Ivory Coast")
lmic_list['Economy'] = lmic_list['Economy'].replace("Gambia, The", "Gambia")
lmic_list['Economy'] = lmic_list['Economy'].replace("Iran, Islamic Rep.", "Iran")

/var/folders/dy/p3vg7l610wx7yc4y2908psy00000gp/T/ipykernel_24001/708080782.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lmic_list['Economy'] = lmic_list['Economy'].replace("Côte d’Ivoire", "Ivory Coast")
/var/folders/dy/p3vg7l610wx7yc4y2908psy00000gp/T/ipykernel_24001/708080782.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lmic_list['Economy'] = lmic_list['Economy'].replace("Gambia, The", "Gambia")
/var/folders/dy/p3vg7l610wx7yc4y2908psy00000gp/T/ipykernel_24001/708080782.py:3: SettingWithCopyW

In [13]:
# Only account for LMIC
only_lmic = False

if only_lmic:
    for year in loc.adjacency_matrices:
        for country in loc.adjacency_matrices[year]['All'].index:
            if country == 'All' or country == 'total':
                continue
            for country2 in loc.adjacency_matrices[year]['All'][country].index:
                if country in list(lmic_list['Economy']) or country2 in list(lmic_list['Economy']):
                    continue
                else:
                    loc.adjacency_matrices[year]['All'][country][country2]=0

In [14]:
loc.load_visjs_nodes_and_edges(node_scaling=0.0015, edge_scaling=0.05, directed=False, threshold=0, font = {"size": 20, "face": "Helvetica Neue"}, node_count='total')

In [15]:
dirname = 'ID_All_Countries2'
loc.to_visjs(vis_name="locations", directed=False, template='geographies/locations.html', dirname=dirname)
loc._to_json("nodes_all.json", loc.vis_nodes)
loc._to_json("edges_all.json", loc.vis_edges)

In [16]:
import os
import shutil

with(open(f'{dirname}/edges.json','r')) as f:
    edges = json.load(f)

os.rename(f'{dirname}/nodes.json', f'{dirname}/nodes_all.json')
shutil.copyfile('geographies/positions.json', f'{dirname}/positions.json')

def edges_to_dict(edges):
    edge_dict = {}
    for e in edges:
        year = e['year']
        _ = e.pop('year')
        if edge_dict.get(year):
            edge_dict[year].append(e)
        else:
            edge_dict[year] = [e]
    return edge_dict


with(open(f'{dirname}/dict_edges_all.json','w')) as f:
        json.dump(edges_to_dict(edges), f)